# Statistical evidence when outcomes arrive late

The first six notebooks separated covariance structure, regression inference, predictive utility, uncertainty, calibration and abrupt temporal drift. Here we investigate gradual concept drift, AR(1) dependence and delayed feedback.

We compare frozen/static, frozen/rolling and periodically refitted/rolling models. All updates are prequential: predictions and decisions are made before the current outcome is available.

A label at deployment step j becomes available only at step j + d + 1, where d is the reporting delay. AR(1) processes and changing coefficients break ordinary exchangeability assumptions; rolling residual quantiles have no automatic coverage guarantee.

In [ ]:
from __future__ import annotations

import os
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from gradual_drift import (
    run_gradual_drift_study,
    aggregate_study,
    paired_delay_penalty,
)

SMOKE = os.environ.get("GRADUAL_SMOKE", "0") == "1"
REPEATS = int(os.environ.get("GRADUAL_REPEATS", "6"))
REGIMES = ("stable", "gradual_drift") if SMOKE else (
    "stable", "gradual_drift", "ar1", "gradual_ar1"
)
SCENARIOS = ("low_variance_signal",) if SMOKE else (
    "low_variance_signal", "high_variance_signal"
)
DELAYS = (0, 3) if SMOKE else (0, 12)
if REPEATS < 1:
    raise ValueError("GRADUAL_REPEATS must be positive")

## Paired simulations and strict observation timing

The generated process has sixteen predictors in eight correlated pairs. Concept drift changes the sign of the conditional response gradually through the last two thirds of deployment. AR(1) scenarios correlate both latent predictors and observation errors, but leave measurement errors independent.

Within a regime, the immediate and delayed-feedback runs use the exact same synthetic stream. The known conditional mean is reserved for evaluation, never for model fitting.

At each step, the learner releases only eligible past labels, updates residual buffers from saved forecasts, optionally refits on eligible labelled history, then predicts and acts. Delaying observations changes what can be learned, not what was actually observed in the simulation.

In [ ]:
intervals, decisions = run_gradual_drift_study(
    repeats=REPEATS, delays=DELAYS,
    regimes=REGIMES, scenarios=SCENARIOS,
    n_fit=80 if SMOKE else 160,
    n_cal=30 if SMOKE else 80,
    n_live=45 if SMOKE else 120,
    rolling_window=20 if SMOKE else 60,
    refit_every=5 if SMOKE else 15,
    refit_window=64 if SMOKE else 120,
)
coverage, economic = aggregate_study(intervals, decisions, bootstrap_draws=300)
display(coverage.round(3))
display(economic.head(15).round(3))

## Interval calibration under lag and dependence

The initial fixed interval is calibrated on a separate historical sample. Marginal split-conformal coverage is justified only for exchangeable calibration and future scores, and not in the dependent or gradually changing regimes.

The rolling interval uses absolute errors from *revealed* past outcomes. Under delayed feedback, its calibration history is necessarily stale. We compare empirical coverage and width by early, transition and late deployment thirds.

In [ ]:
focus = coverage.loc[
    (coverage["scenario"] == "low_variance_signal")
    & (coverage["regime"] == "gradual_drift")
    & (coverage["policy"] == "Refit/rolling")
]
fig, ax = plt.subplots(figsize=(9, 4))
for delay, rows in focus.groupby("delay"):
    ordered = rows.set_index("phase").reindex(["early", "transition", "late"])
    ax.plot(["Early", "Transition", "Late"], ordered["coverage"],
            marker="o", label=f"Delay {delay}")
ax.axhline(0.9, linestyle="--", color="grey", label="Nominal 90%")
ax.set(ylabel="Empirical interval coverage", ylim=(0, 1.02),
       title="Gradual drift: delayed label feedback")
ax.legend()
fig.tight_layout()
plt.show()

## Decision regret of waiting for the label

For intervention cost c, a policy acts when the predicted conditional benefit exceeds c (point threshold), or when its interval lower bound exceeds c (conservative heuristic). Economic regret is evaluated against an oracle that knows the synthetic conditional mean.

The paired quantity is **regret with delayed labels minus regret with immediate labels** on the same generated stream. Positive values mean that delayed feedback led to a larger loss. The mean difference is not guaranteed to be positive in every sample because refitting on fresh but noisy labels can also harm performance.

Bootstrap intervals resample entire independent simulation runs. Consecutive deployment steps are dependent and must not be counted as independent replications.

In [ ]:
penalty = paired_delay_penalty(
    decisions, comparison_delay=DELAYS[-1], baseline_delay=0,
    bootstrap_draws=300,
)
display(penalty.head(20).round(3))
assert intervals["covered"].between(0, 1).all()
assert (decisions["regret"] >= -1e-9).all()

## Interpretation

These comparisons are deliberately synthetic and do not identify causal intervention effects. They separate learning under stale labels from shifts in the true outcome mechanism.

Six independent simulated streams are suitable for exploration, not strong claims of superiority. A future extension should consider observation delays dependent on the outcome itself, richer serial dependence, seasonalities, partial label availability and honest uncertainty estimates over many independent streams. Real-world studies also require data provenance, a temporal splitting protocol and a precisely defined decision cost.